In [1]:
import numpy as np
from FallbackGen import FallbackGen
from TDECalculator import TDECalculator
import gc

In [ ]:
MBH = 1e6
Rp = 6.0
a = 0.0
N = 5000
E = 1.0
Q = 0.0
orbit="rel"

In [3]:
mass_sch = TDECalculator('MAMS1Msun', orbit, MBH, Rp, a, N=N)

In [4]:
sample_sch = mass_sch.rel_whole_star_sample()

In [5]:
radii_sch = sample_sch['rr']

rtde = mass_sch.R_TDE
Lz = mass_sch.mom_kerr_analytic(Rp, a)
E = 1.0
Q = 0

radii = np.where(
    radii_sch <= 0.5,
    rtde - radii_sch * mass_sch.Rstar,
    rtde + radii_sch * mass_sch.Rstar
)

deltaE = mass_sch.Rstar / mass_sch.Rp**2 

i = int(np.random.uniform(0, 1132))
j = int(np.random.uniform(0, 90000))
dE = sample_sch['dEnergy_random'] * deltaE 
dLz = mass_sch.dLz_random
mass_ratio = mass_sch.mass_ratio

In [6]:
dT_sch = FallbackGen(mass_ratio, a, radii, Rp, E, Lz, dE, dLz, N)
np.save('dT_sch.npy', dT_sch.dTs)
gc.collect()

Computing radial periods for 101,880,000 particles ...
  E  range: [0.992114, 1.007885]
  Q  range: [-2.936e-05, 2.936e-05]
  chunk_size = 50,000
  Finding roots (chunked eigensolver) ...
  roots chunk 10189/10189 (100%)
  Bound: 50,944,957 / 101,880,000
  Valid roots: 50,944,957
  Valid Lambda_r: 50,944,957
  Quadrature: 1019 chunks ...
    chunk 1019/1019  (100%)
  Successful T_r: 50,944,957 / 101,880,000


20

In [7]:
def make_plot_dicts(whole_star_sample, dT, delta):
    dE_rand = whole_star_sample["dEnergy_random"]
    dT_rand = dT / delta
    dMass = whole_star_sample["dMass"]

    bins_E = np.linspace(-2.0, 2.0, 1000)
    bins_T = np.logspace(0.0, 6.0, 1000)

    total_mass = np.sum(dMass)
    if not np.isfinite(total_mass) or total_mass <= 0.0:
        raise ValueError("Sample has non-finite or non-positive total mass")

    valid_E = (
        np.isfinite(dE_rand)
        & np.isfinite(dMass)
        & (dE_rand >= bins_E[0])
        & (dE_rand <= bins_E[-1])
    )
    mass_E, edges_E = np.histogram(
        dE_rand[valid_E],
        bins=bins_E,
        weights=dMass[valid_E] / total_mass,
        density=False,
    )
    hist_E = mass_E / np.diff(edges_E)

    valid_T = (
        np.isfinite(dT_rand)
        & np.isfinite(dE_rand)
        & np.isfinite(dMass)
        & (dT_rand >= bins_T[0])
        & (dT_rand <= bins_T[-1])
    )
    mass_T, edges_T = np.histogram(
        dT_rand[valid_T],
        bins=bins_T,
        weights=dMass[valid_T] / total_mass,
        density=False,
    )
    hist_T = mass_T / np.diff(edges_T)

    energy_fraction = np.sum(dMass[valid_E]) / total_mass
    returning_fraction = np.sum(dMass[valid_T]) / total_mass
    assert np.isclose(
        np.sum(hist_E * np.diff(edges_E)), energy_fraction, rtol=1e-12
    )
    assert np.isclose(
        np.sum(hist_T * np.diff(edges_T)), returning_fraction, rtol=1e-12
    )

    print(
        "mass fractions: "
        f"energy range={energy_fraction:.6f}, "
        f"returning/time range={returning_fraction:.6f}"
    )

    energy_plot = {
        "x": 0.5 * (edges_E[:-1] + edges_E[1:]),
        "y": hist_E,
    }
    fallback_plot = {
        "x": 0.5 * (edges_T[:-1] + edges_T[1:]),
        "y": hist_T,
    }
    return energy_plot, fallback_plot

In [8]:
def energy_plots(whole_star_sample): 
    # unpack
    dE_rand   = whole_star_sample['dEnergy_random']      # shape (nr-1, N_Omega)
    dE_unp    = whole_star_sample['dEnergy_unperturbed']
    dT_rand   = whole_star_sample['dT_random']
    dT_unp    = whole_star_sample['dT_unperturbed']
    dMass     = whole_star_sample['dMass']               # same shape

    bins_E = np.linspace(-2.0, 2.0, 1000)
    bins_T = np.logspace(0.0, 6.0, 1000)

    total_mass = np.sum(dMass)
    if not np.isfinite(total_mass) or total_mass <= 0.0:
        raise ValueError("Sample has non-finite or non-positive total mass")

    valid_E = (
        np.isfinite(dE_rand)
        & np.isfinite(dMass)
        & (dE_rand >= bins_E[0])
        & (dE_rand <= bins_E[-1])
    )
    mass_E, edges_E = np.histogram(
        dE_rand[valid_E],
        bins=bins_E,
        weights=dMass[valid_E] / total_mass,
        density=False,
    )
    hist_E = mass_E / np.diff(edges_E)

    valid_T = (
        np.isfinite(dT_rand)
        & np.isfinite(dE_rand)
        & np.isfinite(dMass)
        & (dT_rand >= bins_T[0])
        & (dT_rand <= bins_T[-1])
    )
    mass_T, edges_T = np.histogram(
        dT_rand[valid_T],
        bins=bins_T,
        weights=dMass[valid_T] / total_mass,
        density=False,
    )
    hist_T = mass_T / np.diff(edges_T)

    energy_fraction = np.sum(dMass[valid_E]) / total_mass
    returning_fraction = np.sum(dMass[valid_T]) / total_mass
    assert np.isclose(
        np.sum(hist_E * np.diff(edges_E)), energy_fraction, rtol=1e-12
    )
    assert np.isclose(
        np.sum(hist_T * np.diff(edges_T)), returning_fraction, rtol=1e-12
    )

    print(
        "mass fractions: "
        f"energy range={energy_fraction:.6f}, "
        f"returning/time range={returning_fraction:.6f}"
    )

    energy_plot = {
        "x": 0.5 * (edges_E[:-1] + edges_E[1:]),
        "y": hist_E,
    }
    fallback_plot = {
        "x": 0.5 * (edges_T[:-1] + edges_T[1:]),
        "y": hist_T,
    }

    return energy_plot, fallback_plot

In [9]:
DeltaE = mass_sch.Rstar / mass_sch.Rp**2
DeltaT = 1 / DeltaE**1.5
n_ex = TDECalculator('MAMS1Msun', Rp=Rp, a=0.0, N=N)

In [10]:
whole_star_sample4 = n_ex.whole_star_sample()

In [11]:
n_E, n_T = energy_plots(whole_star_sample4)

mass fractions: energy range=1.000000, returning/time range=0.498528


In [12]:
rel_sch_E, rel_sch_T = make_plot_dicts(sample_sch, dT_sch.dTs, DeltaT)

mass fractions: energy range=1.000000, returning/time range=0.498939


In [13]:
import json

adden = "m1_rp6"

with open(f"Fallback_Data/rel_sch_e_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_sch_E.items()}, f)
with open(f"Fallback_Data/rel_sch_t_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_sch_T.items()}, f)

adden = "m1_rp6"

with open(f"Fallback_Data/n_e_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in n_E.items()}, f)
with open(f"Fallback_Data/n_t_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in n_T.items()}, f)
